In [132]:
#!python -m pip install --upgrade pip
#%pip install pandas matplotlib seaborn scikit-learn openpyxl tensorflow xgboost aif360
#%pip install "aif360[Reductions, inFairness]"

In [133]:
import json
import pandas as pd
from pprint import pprint
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier

import tensorflow as tf
from tensorflow.keras.models import Sequential # type: ignore
from tensorflow.keras.layers import Dense, BatchNormalization # type: ignore

from aif360.datasets import StandardDataset
from fairlearn.preprocessing import CorrelationRemover
from aif360.algorithms.inprocessing import GerryFairClassifier
from fairlearn.postprocessing import ThresholdOptimizer

random_seed = 15

In [134]:
PATH = 'C:/Users/andre/Desktop/ProjectWork_AEQUITAS_AKKODIS/'
df = (
    pd.read_excel(PATH + 'data/Dataset_Preprocessed.xlsx')
)
df.head()

,Technical Skills,Comunication,Maturity,Dynamism,Mobility,English,Candidate State_encoded,Event_Feedback_encoded,Residence City_encoded,Residence Province_encoded,...,Sector_encoded,Job Family Hiring_encoded,Job Title Hiring_encoded,Overall_encoded,Minimum Ral_encoded,Ral Maximum_encoded,Study Level_encoded,Current Ral_encoded,Expected Ral_encoded,Status_encoded
0,3,2,3,3,2,4,6,11,689,102,...,1,3,5,5,7,9,6,4,5,1
1,2,2,2,2,2,3,4,10,218,8,...,0,4,8,0,1,1,0,5,6,0
2,2,2,2,2,2,3,3,9,157,22,...,2,4,8,0,1,1,0,1,1,1
3,2,2,2,2,2,3,3,11,636,57,...,13,4,8,0,1,1,0,3,3,1
4,3,2,2,3,1,1,2,11,39,98,...,1,4,8,3,1,1,0,1,5,0


In [135]:
with open(PATH + 'data/encoding_mappings.json', 'r') as f:
    encoding_mappings = json.load(f)
pprint(encoding_mappings)

{'Age Range': {'20 - 25 years': 1,
               '26 - 30 years': 2,
               '31 - 35 years': 3,
               '36 - 40 years': 4,
               '40 - 45 years': 5,
               '< 20 years': 0,
               '> 45 years': 6},
 'Candidate State': {'Economic proposal': 5,
                     'First contact': 1,
                     'Hired': 6,
                     'Imported': 0,
                     'In selection': 2,
                     'QM': 3,
                     'Vivier': 4},
 'Current Ral': {'+ 50 K': 18,
                 '- 20 K': 2,
                 '20-22 K': 3,
                 '22-24 K': 4,
                 '24-26 K': 5,
                 '26-28 K': 6,
                 '28-30 K': 7,
                 '30-32 K': 8,
                 '32-34 K': 9,
                 '34-36 K': 10,
                 '36-38 K': 11,
                 '38-40 K': 12,
                 '40-42 K': 13,
                 '42-44 K': 14,
                 '44-46 K': 15,
                 '46-48 K': 16

## Train

In [136]:
target = 'Status_encoded'
sensitive = 'Sex_encoded'

### Dataset Preparation

In [137]:
df = shuffle(df, random_state=random_seed)

X = df.copy()
y = df[target]
s = df[sensitive]
X_train_split, X_test_split, y_train_split, y_test_split, s_train_split, s_test_split = train_test_split(X, y, s, test_size=0.2, random_state=random_seed, stratify=y)

In [138]:
train_df = X_train_split.copy()
train_df[target] = y_train_split.values
train_df[sensitive] = s_train_split.values

train_ds = StandardDataset(
    train_df,
    label_name=target, # the column with labels (0/1)
    favorable_classes=[1], # value considered positive
    protected_attribute_names=[sensitive], # or ['race'], etc.
    privileged_classes=[[1]] # e.g., male if 1 = male
)

In [139]:
test_df = X_test_split.copy()
test_df[target] = y_test_split.values
test_df[sensitive] = s_test_split.values

test_ds = StandardDataset(
    test_df,
    label_name=target, # the column with labels (0/1)
    favorable_classes=[1], # value considered positive
    protected_attribute_names=[sensitive], # or ['race'], etc.
    privileged_classes=[[1]] # e.g., male if 1 = male
)

In [140]:
predictions = {}

### Pre-Processing

In [141]:
cr = CorrelationRemover(sensitive_feature_ids=[sensitive], alpha=1)

X_train_cr = cr.fit_transform(train_df)
X_train_cr_df = pd.DataFrame(X_train_cr)

X_test_cr = cr.transform(test_df)
X_test_cr_df = pd.DataFrame(X_test_cr)

In [142]:
def create_model(seed, input_dim):
    tf.random.set_seed(seed)
    model = Sequential()
    model.add(Dense(128, input_dim=input_dim, activation='relu'))
    model.add(BatchNormalization())
    model.add(Dense(128, activation='relu'))
    model.add(BatchNormalization())
    model.add(Dense(128, activation='relu'))
    model.add(BatchNormalization())
    model.add(Dense(64, activation='relu'))
    model.add(BatchNormalization())
    model.add(Dense(1, activation='sigmoid'))

    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

models = {
    'Logistic Regression': LogisticRegression(),
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeClassifier(),
    'Naive Bayes': GaussianNB(),
    'XGBoost': XGBClassifier(),
    'KNN': KNeighborsClassifier(),
    'Neural Network': create_model(random_seed, X_train_cr_df.shape[1]),
}

In [143]:
for model_name, model in models.items():
    print(f"Training {model_name}...")
    if model_name == 'Neural Network':
        model.fit(X_train_cr_df, y_train_split, epochs=10, batch_size=32, verbose=0)
        predictions[f"{model_name}_preprocessed_cr"] = model.predict(X_test_cr_df).flatten()
    else:
        model.fit(X_train_cr_df, y_train_split)
        predictions[f"{model_name}_preprocessed_cr"] = model.predict(X_test_cr_df)

    if model_name in ['Linear Regression', 'XGBoost', 'Neural Network']:
        predictions[f"{model_name}_preprocessed_cr"] = (predictions[f"{model_name}_preprocessed_cr"] > 0.5).astype(int)
    
    print(f"{model_name} trained.")
    temp = predictions[f"{model_name}_preprocessed_cr"]
    print(f"Preprocessed predictions for {model_name}: {temp}")
    

Training Logistic Regression...


c:\Users\andre\Desktop\ProjectWork_AEQUITAS_AKKODIS\.venv\lib\site-packages\sklearn\linear_model\_logistic.py:460: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Logistic Regression trained.
Preprocessed predictions for Logistic Regression: [0 0 0 0 0 0 0 1 1 1 0 1 1 0 0 0 1 1 0 1 0 0 1 0 0 1 0 1 1 1 0 1 1 1 0 0 0
 0 0 0 0 0 1 0 1 0 0 0 1 0 0 0 1 1 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 1 1 0 1
 1 0 0 1 0 1 0 1 0 1 1 0 1 0 0 0 1 0 0 0 1 1 0 0 0 0 1 1 0 0 1 1 0 0 0 1 0
 0 0 0 0 0 1 0 0 0 1 0 0 0 1 0 1 0 0 1 0 1 0 0 1 0 0 0 0 1 1 0 0 0 0 0 1 1
 0 0 1 0 0 0 0 0 0 1 1 0 0 1 0 1 0 0 0 1 1 0 1 0 0 1 0 0 0 0 0 0 0 0 0 0 1
 1 0 0 0 0 1 1 1 0 1 0 1 1 0 0 1 0 1 0 0 0 0 1 1 0 0 0 0 0 0 0 1 0 1 0 0 0
 0 0 1 0 0 0 0 0 0 0 0 0 0 0 1 1 0 0 1 0 1 0 0 0 0 0 0 1 1 0 0 0 0 0 1 0 0
 0 0 0 0 1 0 1 1 0 0 0 1 0 1 0 0 1 1 0 1 0 1 1 0 0 0 0 1 0 1 1 0 1 0 1 0 0
 0 0 1 1 0 1 1 1 0 0 0 0 0 0 0 1 0 0 0 0 0 1 0 0 0 1 1 0 0 0 0 0 0 0 1 1 0
 0 0 0 0 0 1 1 1 0 0 0 0 0 1 0 0 0 0 0 1 0 0 1 0 0 0 0 0 0 1 0 0 0 0 0 0 1
 0 0 1 0 0 0 1 0 1 0 0 0 1 1 0 0 1 1 0 0 1 0 1 0 0 1 0 0 1 0 1 0 0 1 0 0 1
 0 1 0 0 0 0 0 1 1 1 1 0 1 0 0 0 1 1 0 1 1 0 1 0 0 1 0 0 1 1 1 0 0 1 0 0 0
 1 0 1 1 0 1 1 0 0 0 

### In-Processing

In [144]:
models = {
    'Linear Regression': LinearRegression(),
}

In [145]:
for model_name, model in models.items():
    gfc = GerryFairClassifier(
        C=10,
        gamma=0.01,
        fairness_def='FP',
        max_iters=10,
        printflag=False,
        heatmapflag=False,
        heatmap_iter=10,
        heatmap_path='.',
        predictor=model
    )
    gfc.fit(train_ds)
    pred_gfc = gfc.predict(test_ds)
    predictions[f"{model_name}_inprocessed_gfc"] = pred_gfc.labels.ravel()
    temp = predictions[f"{model_name}_inprocessed_gfc"]
    print(f"Inprocessed predictions for {model_name}: {temp}")


Inprocessed predictions for Linear Regression: [0 0 0 0 0 0 0 0 1 1 0 1 1 0 0 0 0 0 0 1 0 0 1 0 0 1 0 1 1 0 0 1 0 1 0 0 0
 0 0 0 0 0 1 0 1 0 0 0 0 0 0 0 1 1 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 1
 0 0 0 1 0 0 0 0 0 0 1 0 1 0 0 0 1 0 0 0 1 0 0 0 0 0 0 0 0 0 1 0 0 0 0 1 0
 0 0 0 0 0 1 0 0 0 1 0 0 0 1 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 1
 0 0 1 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 1 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 1
 1 0 0 0 0 1 1 1 0 0 0 0 0 0 0 0 0 1 0 0 0 0 1 0 0 0 0 0 0 0 0 1 0 0 0 0 0
 0 0 1 0 0 0 0 0 0 0 0 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0
 0 0 0 0 0 0 0 1 0 0 0 1 0 0 0 0 1 1 0 1 0 1 1 0 0 0 0 1 0 1 1 0 1 0 1 0 0
 0 0 0 1 0 1 1 0 0 0 0 0 0 0 0 1 0 0 0 0 0 1 0 0 0 1 0 0 0 0 0 0 0 0 1 1 0
 0 0 0 0 0 0 1 1 0 0 0 0 0 1 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 1 0 0 0 0 0 0 1
 0 0 1 0 0 0 1 0 0 0 0 0 1 0 0 0 0 0 0 0 1 0 1 0 0 0 0 0 1 0 1 0 0 0 0 0 0
 0 1 0 0 0 0 0 0 0 1 1 0 1 0 0 0 0 0 0 0 1 0 0 0 0 1 0 0 0 1 1 0 0 0 0 0 0
 0 0 1 1 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 1 1 0 1 1 1 

### Post-Processing

In [146]:
def create_model(seed, input_dim):
    tf.random.set_seed(seed)
    model = Sequential()
    model.add(Dense(128, input_dim=input_dim, activation='relu'))
    model.add(BatchNormalization())
    model.add(Dense(128, activation='relu'))
    model.add(BatchNormalization())
    model.add(Dense(128, activation='relu'))
    model.add(BatchNormalization())
    model.add(Dense(64, activation='relu'))
    model.add(BatchNormalization())
    model.add(Dense(1, activation='sigmoid'))

    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

models = {
    'Logistic Regression': LogisticRegression(),
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeClassifier(),
    'Naive Bayes': GaussianNB(),
    'XGBoost': XGBClassifier(),
    'KNN': KNeighborsClassifier(),
    'Neural Network': create_model(random_seed, train_df.shape[1]),
}

In [147]:
for model_name, model in models.items():
    print(f"Training {model_name}...")
    if model_name == 'Neural Network':
        model.fit(train_df, y_train_split, epochs=10, batch_size=32, verbose=0)
        predictions[f"{model_name}_postprocessed_to"] = model.predict(test_df).flatten()
    else:
        model.fit(train_df, y_train_split)
        predictions[f"{model_name}_postprocessed_to"] = model.predict(test_df)

    print(f"{model_name} trained.")

Training Logistic Regression...
Logistic Regression trained.
Training Linear Regression...
Linear Regression trained.
Training Decision Tree...
Decision Tree trained.
Training Naive Bayes...
Naive Bayes trained.
Training XGBoost...


c:\Users\andre\Desktop\ProjectWork_AEQUITAS_AKKODIS\.venv\lib\site-packages\sklearn\linear_model\_logistic.py:460: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


XGBoost trained.
Training KNN...
KNN trained.
Training Neural Network...
17/17 [==============================] - 1s 5ms/step
Neural Network trained.


In [148]:
for model_name, model in models.items():
    to = ThresholdOptimizer(
        estimator=model,
        constraints='demographic_parity',
        objective='accuracy_score',
        prefit=True,
        grid_size=1000,
        flip=False,
        predict_method='predict'
    )
    to.fit(train_df, y_train_split, sensitive_features=s_train_split)
    pred_to = to.predict(test_df, sensitive_features=s_test_split)
    predictions[f"{model_name}_postprocessed_to"] = pred_to.ravel()
    temp = predictions[f"{model_name}_postprocessed_to"]
    print(f"Postprocessed predictions for {model_name}: {temp}")

Postprocessed predictions for Logistic Regression: [0 0 0 0 0 0 0 1 1 0 0 1 1 0 0 0 1 1 0 1 0 0 1 0 0 1 0 1 1 1 0 0 0 1 0 0 0
 0 0 0 0 0 1 0 1 0 0 0 1 0 0 0 1 0 0 0 1 1 1 0 0 0 0 0 0 0 0 0 0 0 1 1 0 1
 1 0 0 1 1 1 0 1 0 1 1 0 1 0 0 0 1 0 0 0 1 1 0 0 0 0 1 1 0 0 1 1 0 0 0 1 0
 0 0 0 0 0 1 0 0 0 1 0 0 0 1 0 1 0 0 1 0 1 0 0 1 0 0 0 0 1 1 0 0 0 0 0 1 1
 1 0 1 1 0 0 0 0 0 1 1 0 0 1 0 1 0 0 0 1 1 0 1 0 0 1 0 1 0 0 0 0 0 0 0 0 1
 1 1 0 0 0 1 1 1 0 1 0 1 1 0 1 1 0 1 0 0 0 0 1 1 0 0 0 0 0 0 0 0 1 1 0 0 0
 0 1 1 0 0 0 0 0 0 0 0 0 0 0 1 1 1 0 0 0 1 0 0 0 0 0 0 1 1 0 0 0 0 0 1 0 0
 0 0 0 0 1 0 0 1 0 0 0 1 0 1 0 0 1 1 0 1 0 1 1 0 0 0 0 1 0 1 1 0 1 0 1 0 0
 0 0 1 1 0 1 1 1 0 0 0 0 0 0 0 1 0 0 0 0 0 1 1 0 0 1 1 0 0 0 0 0 0 0 1 1 0
 0 0 0 0 0 1 1 1 0 1 0 0 0 1 0 0 0 0 0 1 0 0 1 0 0 0 0 0 0 1 0 0 0 0 0 0 1
 0 0 1 0 0 0 1 0 1 0 0 0 1 1 0 0 1 1 0 0 1 0 1 0 1 1 0 0 1 0 1 0 0 1 0 0 1
 0 1 0 0 0 0 0 1 1 1 1 0 1 0 0 0 1 1 0 1 1 0 1 0 0 1 0 0 1 1 1 0 0 1 0 0 0
 1 0 1 1 0 1 1 0 0 0 1 0 1 1 1 0 0 0 0 0 1 1 0 1 

c:\Users\andre\Desktop\ProjectWork_AEQUITAS_AKKODIS\.venv\lib\site-packages\fairlearn\postprocessing\_threshold_optimizer.py:309: UserWarning: The value of `prefit` is `True`, but `check_is_fitted` raised `NotFittedError` on the base estimator.

If the provided base estimator has been fitted, this could mean that (1) its implementation does not conform to the sklearn estimator API, or (2) the enclosing ThresholdOptimizer has been cloned (for instance by `sklearn.model_selection.cross_validate`).

In case (1), please file an issue with the base estimator developers, but continue to use the enclosing ThresholdOptimizer with `prefit=True`. In case (2), please use `prefit=False`.
  warn(BASE_ESTIMATOR_NOT_FITTED_WARNING.format(type(self).__name__))


 1/17 [>.............................] - ETA: 1s

c:\Users\andre\Desktop\ProjectWork_AEQUITAS_AKKODIS\.venv\lib\site-packages\fairlearn\postprocessing\_interpolated_thresholder.py:107: UserWarning: The value of `prefit` is `True`, but `check_is_fitted` raised `NotFittedError` on the base estimator.

If the provided base estimator has been fitted, this could mean that (1) its implementation does not conform to the sklearn estimator API, or (2) the enclosing InterpolatedThresholder has been cloned (for instance by `sklearn.model_selection.cross_validate`).

In case (1), please file an issue with the base estimator developers, but continue to use the enclosing InterpolatedThresholder with `prefit=True`. In case (2), please use `prefit=False`.
  warn(BASE_ESTIMATOR_NOT_FITTED_WARNING.format(type(self).__name__))


17/17 [==============================] - 0s 8ms/step
Postprocessed predictions for Neural Network: [0 0 0 0 0 0 0 1 1 1 0 1 1 0 0 0 1 1 0 1 0 0 1 0 0 1 0 1 1 1 0 1 0 1 0 0 0
 0 0 0 0 0 1 0 1 0 0 0 1 0 0 0 1 1 0 0 1 1 1 0 0 0 0 0 0 0 0 0 0 0 1 1 0 1
 1 0 0 1 0 1 0 1 0 1 1 0 1 0 0 0 1 0 0 0 1 1 0 0 0 1 1 1 0 0 1 1 0 0 0 1 0
 0 0 0 0 0 1 0 0 0 1 0 0 0 1 0 1 0 0 1 0 1 0 0 1 0 0 0 0 1 1 0 0 0 0 0 1 1
 0 0 1 1 0 0 0 0 1 1 1 0 0 1 0 1 0 0 0 1 1 0 1 0 0 1 0 1 0 0 0 0 0 0 0 1 1
 1 1 0 0 0 1 1 1 0 1 0 1 1 1 1 1 0 1 0 0 0 0 1 1 0 0 0 0 0 0 0 1 1 1 0 0 0
 0 1 1 0 0 0 0 0 0 0 0 0 0 0 1 1 0 0 1 0 1 0 0 0 0 0 0 1 1 0 0 0 0 0 1 0 0
 0 0 0 0 1 0 0 1 0 0 0 1 0 1 0 0 1 1 0 1 0 1 1 0 0 0 0 1 0 1 1 0 1 0 1 0 0
 0 0 1 1 0 1 1 1 0 0 0 0 0 0 0 1 0 0 0 0 0 1 1 0 0 1 1 0 0 0 0 0 0 0 1 1 0
 0 0 0 0 0 1 1 1 0 1 0 0 0 1 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 1 0 0 0 0 0 0 1
 0 0 1 0 0 0 1 0 1 0 0 0 1 1 0 0 1 1 0 0 1 0 1 0 1 1 0 0 1 0 1 0 0 1 0 0 1
 0 1 0 0 0 0 0 1 1 1 1 0 1 0 0 0 1 1 0 1 1 0 1 0 0 1 0 0 1 0 1 0 0 1 0 0 0
 

## Save

In [149]:
reference = {}
for model_name, preds in predictions.items():
    reference[model_name] = y_test_split.values

data = pd.DataFrame({
    'predictions': predictions,
    'reference':   reference,
})
data.head(50)

,predictions,reference
Logistic Regression_preprocessed,"[0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, ..."
Linear Regression_preprocessed,"[0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, ..."
Decision Tree_preprocessed,"[0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, ..."
Naive Bayes_preprocessed,"[0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, ..."
XGBoost_preprocessed,"[0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, ..."
KNN_preprocessed,"[0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, ..."
Neural Network_preprocessed,"[0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, ..."
Linear Regression_inprocessed_gfc,"[0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, ..."
Logistic Regression_postprocessed_to,"[0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, ..."
Linear Regression_postprocessed_to,"[0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, ..."


In [150]:
data.to_excel(PATH + 'data/predictions.xlsx', index=False)